In [ ]:
import os
import pandas as pd
import numpy as np

RAW_PATH = "data/raw/"
PROCESSED_PATH = "data/processed/"

os.makedirs(PROCESSED_PATH, exist_ok=True)

all_data = []

# =====================================================
# 1️⃣ LOAD NIFTY 50 (Market Context)
# =====================================================

nifty = pd.read_csv(
    os.path.join(RAW_PATH, "NIFTY50.csv"),
    index_col=0,
    parse_dates=True
)

# Convert all columns to numeric
nifty = nifty.apply(pd.to_numeric, errors='coerce')

# Sort & clean
nifty = nifty.sort_index()
nifty = nifty.dropna()

# Create NIFTY Return
nifty['NIFTY_Return'] = nifty['Close'].pct_change(fill_method=None)

print("✅ NIFTY Loaded Successfully")

# =====================================================
# 2️⃣ PROCESS ALL STOCK FILES
# =====================================================

for file in os.listdir(RAW_PATH):

    if file.endswith(".csv") and file != "NIFTY50.csv":

        ticker = file.replace(".csv", "")
        print(f"Processing {ticker}...")

        # 🔹 Read stock file correctly
        df = pd.read_csv(
            os.path.join(RAW_PATH, file),
            index_col=0,
            parse_dates=True
        )

        # Convert to numeric
        df = df.apply(pd.to_numeric, errors='coerce')

        # Cleaning
        df = df.sort_index()
        df = df.dropna()

        # =====================================================
        # FEATURE ENGINEERING
        # =====================================================

        # Daily Return
        df['Daily_Return'] = df['Close'].pct_change()

        # Moving Averages
        df['MA7'] = df['Close'].rolling(7).mean()
        df['MA30'] = df['Close'].rolling(30).mean()

        # Rolling Volatility
        df['Volatility_7'] = df['Daily_Return'].rolling(7).std()
        df['Volatility_14'] = df['Daily_Return'].rolling(14).std()

        # RSI (14)
        delta = df['Close'].diff()
        gain = delta.where(delta > 0, 0).rolling(14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
        rs = gain / loss
        df['RSI'] = 100 - (100 / (1 + rs))

        # MACD
        ema12 = df['Close'].ewm(span=12, adjust=False).mean()
        ema26 = df['Close'].ewm(span=26, adjust=False).mean()
        df['MACD'] = ema12 - ema26
        df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
        df['MACD_Hist'] = df['MACD'] - df['MACD_Signal']

        # Merge NIFTY Return
        df = df.merge(
            nifty['NIFTY_Return'],
            left_index=True,
            right_index=True,
            how='left'
        )

        # Target (Next Day Return)
        df['Target'] = df['Daily_Return'].shift(-1)
       # Based on today’s market data and indicators, what will be tomorrow’s return?

        # Drop NaNs after feature creation
        df = df.dropna()

        # Add Ticker Column
        df['Ticker'] = ticker

        all_data.append(df)

# =====================================================
# 3️⃣ COMBINE ALL STOCKS
# =====================================================

final_df = pd.concat(all_data)

# Save final dataset
final_df.to_csv(os.path.join(PROCESSED_PATH, "final_dataset.csv"))

print("🎉 All stocks processed successfully!")
print("Final dataset shape:", final_df.shape)
